In [ ]:
from dotenv import load_dotenv

load_dotenv()

from pathlib import Path

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import soundfile as sf
import wandb

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L

import torchvision
import torchmetrics
from pydantic import BaseModel

In [ ]:
run_name = None

DATA_DIR = Path("data")
BIRDCLEF_DIR = DATA_DIR / "birdclef"
TRAIN_SOUNDSCAPES_DIR = BIRDCLEF_DIR / "train_soundscapes"
PERCH_MODEL_DIR = DATA_DIR / "perch"
PERCH_EMBEDDING_DIM = 1536

DURATION = 60
SAMPLE_RATE = 32000
NUM_CLASSES = 234


class Config(BaseModel):
    max_epochs: int = 10
    batch_size: int = 256
    lr: float = 1e-3


config = Config()

In [ ]:
perch_classes = pd.read_csv(PERCH_MODEL_DIR / "assets" / "labels.csv")[
    "inat2024_fsd50k"
].values

taxonomy = pd.read_csv(BIRDCLEF_DIR / "taxonomy.csv")

scientific_name_by_primary_label = taxonomy.set_index("primary_label")[
    "scientific_name"
]

birdclef_classes = pd.read_csv(BIRDCLEF_DIR / "taxonomy.csv")["scientific_name"].values

train_soundscapes_labels_df = pd.read_csv(BIRDCLEF_DIR / "train_soundscapes_labels.csv")
train_soundscapes_labels = (
    train_soundscapes_labels_df["primary_label"]
    .str.split(";")
    .explode()
    .map(scientific_name_by_primary_label)
)
train_soundscapes_classes = train_soundscapes_labels.unique()

train_audios_classes = pd.read_csv(BIRDCLEF_DIR / "train.csv")[
    "scientific_name"
].unique()

In [ ]:
nfiles_per_class = (
    train_soundscapes_labels_df.assign(
        primary_label=train_soundscapes_labels_df["primary_label"].str.split(";")
    )
    .explode("primary_label")
    .groupby("primary_label")["filename"]
    .nunique()
)
nfiles_per_class.index = nfiles_per_class.index.map(scientific_name_by_primary_label)

untrained_classes = birdclef_classes[~birdclef_classes.isin(perch_classes)]
nfiles_per_class.quantile([0.2, 0.5, 0.7, 0.9])

In [ ]:
import zlib


def split_soundscapes(soundscapes: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    folds = np.array(
        [zlib.crc32(filename.encode()) % 10 for filename in soundscapes["filename"]]
    )
    train_mask = folds != 0

    return soundscapes[train_mask], soundscapes[~train_mask]


def label_distribution(df: pd.DataFrame):
    return df["primary_label"].str.split(";").explode().value_counts(normalize=True)


soundscapes_df = pd.read_csv(BIRDCLEF_DIR / "train_soundscapes_labels.csv")
train_df, val_df = split_soundscapes(soundscapes_df)

In [ ]:
[len(label_distribution(v)) for v in (soundscapes_df, train_df, val_df)]

In [ ]:
plt.subplot(1, 2, 1)
label_distribution(train_df).sort_values().plot.bar(figsize=(20, 5))
plt.subplot(1, 2, 2)
label_distribution(val_df).sort_values().plot.bar()

In [ ]:
# val_df['primary_label'].str.split(';').explode().unique()

In [ ]:
class ClassifierModel(nn.Module):
    def __init__(
        self,
        emb_dim: int = PERCH_EMBEDDING_DIM,
        proj_dim: int = 256,
        n_classes: int = NUM_CLASSES,
    ) -> None:
        super().__init__()
        self.proj = nn.Linear(emb_dim, proj_dim)
        self.classifier = nn.Linear(proj_dim + proj_dim + proj_dim, n_classes)
        self.attention = nn.Linear(proj_dim, n_classes)

    def forward(self, emb, whole: bool = False):
        """
        Input:
            emb: (batch, frames, emb_dim)

        Output:
            logits: (batch, frames, classes)
        """

        p = self.proj(emb)

        global_context = torch.sum(p, dim=1, keepdim=True).expand(*p.shape)

        near_context = torch.zeros_like(p)
        near_context[:, :-1, :] += p[:, 1:, :]
        near_context[:, 1:, :] += p[:, :-1, :]

        logits = self.classifier(
            torch.cat(
                [
                    p,
                    near_context,
                    global_context,
                ],
                dim=-1,
            )
        )

        if whole:
            weights = self.attention(p)
            weights = torch.softmax(weights, dim=1)
            logits = torch.sum(logits * weights, dim=1)

        return logits

In [ ]:
class_idx_by_primary_label = {v: i for i, v in enumerate(taxonomy["primary_label"])}


def get_birdclef_class_idx(primary_label: str) -> int:
    return class_idx_by_primary_label[primary_label]


def time_string_to_seconds(time_string: str):
    h, m, s = [int(v) for v in time_string.split(":")]
    return h * 3600 + m * 60 + s


def parse_soundscapes_labels(df: pd.DataFrame):
    files = {}
    seen = set()

    for _, row in df.iterrows():
        filename = row["filename"]
        if filename not in files:
            files[filename] = []

        start_time = time_string_to_seconds(row["start"])
        end_time = time_string_to_seconds(row["end"])

        k = (filename, start_time, end_time)
        if k in seen:
            continue

        seen.add(k)

        labels = row["primary_label"].split(";")

        files[filename].append((start_time, end_time, labels))

    return files


class Embeddings:
    def __init__(self, embeddings):
        self.embeddings = embeddings

    def get_embedding(
        self, filename: str, start_time: int, end_time: int
    ) -> np.ndarray:
        return self.embeddings[(filename, start_time, end_time)]


class TrainAudioDataset(torch.utils.data.Dataset):
    def __init__(self, birdclef_dir: str, embeddings: Embeddings):
        super().__init__()

        files = pd.read_csv(Path(birdclef_dir / "train.csv"))["filename"].to_list()
        files = [
            file
            for file in files
            if sf.info(Path(birdclef_dir) / "train_audio" / file).duration <= DURATION
        ]
        self.files = files
        self.embeddings = embeddings

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx: int):
        filepath = self.files[idx]

        NUM_SEGMENTS = 12

        emb = np.array(
            [
                self.embeddings.get_embedding(
                    "train_audio/" + filepath, i * 5, i * 5 + 5
                )
                for i in range(NUM_SEGMENTS)
            ]
        )

        labels = np.zeros((NUM_CLASSES,), dtype=np.long)
        primary_label = filepath[: filepath.find("/")]
        labels[get_birdclef_class_idx(primary_label)] = 1

        emb = torch.from_numpy(emb)
        labels = torch.from_numpy(labels).long()
        return emb, labels


class SoundscapesDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, embeddings: Embeddings):
        super().__init__()

        self.files = list(parse_soundscapes_labels(df).items())
        self.embeddings = embeddings

    def get_class_indices(self) -> list[int]:
        used_labels = set()

        for _, segments in self.files:
            for _, _, labels in segments:
                for label in labels:
                    used_labels.add(label)

        return sorted([get_birdclef_class_idx(label) for label in used_labels])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx: int):
        filename, segments = self.files[idx]

        NUM_SEGMENTS = 12

        emb = np.array(
            [
                self.embeddings.get_embedding(
                    "train_soundscapes/" + filename, i * 5, i * 5 + 5
                )
                for i in range(NUM_SEGMENTS)
            ]
        )

        labels = np.zeros((NUM_SEGMENTS, NUM_CLASSES), dtype=np.long)
        attention_mask = np.zeros((NUM_SEGMENTS,), dtype=np.long)

        for s, _, primary_labels in segments:
            assert s % 5 == 0
            labels_idx = s // 5

            attention_mask[labels_idx] = 1

            for l in primary_labels:
                labels[labels_idx, get_birdclef_class_idx(l)] = 1

        labels = np.array(labels)

        emb = torch.from_numpy(emb)
        labels = torch.from_numpy(labels).long()
        attention_mask = torch.from_numpy(attention_mask).long()
        return emb, labels, attention_mask


class BirdClefDataModule(L.LightningDataModule):
    def __init__(
        self, birdclef_dir: str, embeddings: Embeddings, batch_size: int = 1
    ) -> None:
        super().__init__()

        self.batch_size = batch_size

        soundscapes_df = pd.read_csv(
            Path(birdclef_dir) / "train_soundscapes_labels.csv"
        )
        train_df, val_df = split_soundscapes(soundscapes_df)

        self.train_ds = SoundscapesDataset(train_df, embeddings)
        self.val_ds = SoundscapesDataset(val_df, embeddings)
        self.train_audio_ds = TrainAudioDataset(birdclef_dir, embeddings)

        # To be used later for evaluation to determine which classes to retain/exclude for metrics calculation.
        self.val_class_indices = self.val_ds.get_class_indices()

    def train_dataloader(self):
        return [
            torch.utils.data.DataLoader(
                self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=4
            ),
            torch.utils.data.DataLoader(
                self.train_audio_ds,
                batch_size=self.batch_size,
                shuffle=True,
                num_workers=4,
            ),
        ]

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=4
        )


class FocalLoss(nn.Module):
    def __init__(
        self,
        alpha: float = 0.25,
        gamma: float = 2,
        reduction: str = "mean",
    ):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        return torchvision.ops.focal_loss.sigmoid_focal_loss(
            inputs=inputs,
            targets=targets,
            alpha=self.alpha,
            gamma=self.gamma,
            reduction=self.reduction,
        )


class MaskedBCEWithLogitsLoss(nn.Module):
    def __init__(self):
        super().__init__()

        self.bce = FocalLoss()

    def forward(self, input, target, attention_mask):
        return self.bce(input[attention_mask], target[attention_mask])


class TrainingModule(L.LightningModule):
    def __init__(
        self, weight_decay: float = 1e-4, lr: float = 1e-3, warmup_pct: float = 0.1
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        self.model = ClassifierModel()
        self.bce = FocalLoss()
        self.criterion = MaskedBCEWithLogitsLoss()

        self.val_class_indices = None
        self.val_metrics = None

    def setup(self, stage: str):
        self.val_class_indices = self.trainer.datamodule.val_class_indices
        num_labels = len(self.val_class_indices)

        self.val_metrics = torchmetrics.MetricCollection(
            {
                "aucroc": torchmetrics.classification.MultilabelAUROC(num_labels),
                "aucroc_micro": torchmetrics.classification.MultilabelAUROC(
                    num_labels, average="micro"
                ),
                "map": torchmetrics.classification.MultilabelAveragePrecision(
                    num_labels
                ),
                "f1": torchmetrics.classification.MultilabelF1Score(num_labels),
                "accuracy": torchmetrics.classification.MultilabelAccuracy(num_labels),
            },
            prefix="val/",
        )

    def training_step(self, batch, batch_idx):
        emb, labels, attention_mask = batch[0]

        logits = self.model(emb)
        logits = logits.reshape(-1, logits.shape[-1])
        labels = labels.reshape(-1, labels.shape[-1])
        loss_1 = self.criterion(logits, labels.float(), attention_mask)

        emb, labels = batch[1]
        logits = self.model(emb, whole=True)
        loss_2 = self.bce(logits, labels.float())

        loss = loss_1 + loss_2

        self.log("train/loss", loss)
        self.log("train/loss_audios", loss_2)
        return loss

    def validation_step(self, batch, batch_idx):
        emb, labels, attention_mask = batch

        logits = self.model(emb)
        logits = logits.reshape(-1, logits.shape[-1])
        labels = labels.reshape(-1, labels.shape[-1])
        loss = self.criterion(logits, labels.float(), attention_mask)

        self.log("val/loss", loss)

        self._update_val_metrics(logits, labels)

    def _update_val_metrics(self, logits, labels):
        assert self.val_metrics is not None
        assert self.val_class_indices is not None

        logits = logits[:, self.val_class_indices]
        labels = labels[:, self.val_class_indices]

        self.val_metrics.update(logits, labels)

    def on_validation_epoch_end(self):
        assert self.val_metrics is not None

        self.log_dict(self.val_metrics.compute())
        self.val_metrics.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )

        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=self.hparams.lr,
            total_steps=self.trainer.estimated_stepping_batches,
            pct_start=self.hparams.warmup_pct,
            anneal_strategy="cos",
            div_factor=25.0,
            final_div_factor=1000.0,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
            },
        }


In [ ]:
# Compute and cache perch embeddings

EMBEDDINGS_CACHE_PATH = "embeddings_cache.npz"

if not os.path.exists(EMBEDDINGS_CACHE_PATH):
    import tensorflow as tf

    class PerchModel:
        def __init__(self, saved_model_path: str):
            model = tf.saved_model.load(saved_model_path)
            self.infer_fn = model.signatures["serving_default"]
            self.classes = pd.read_csv(
                Path(saved_model_path) / "assets" / "labels.csv"
            )["inat2024_fsd50k"].tolist()

        def predict(self, wave: np.ndarray):
            res = self.infer_fn(inputs=wave)
            res["label"] = tf.nn.softmax(res["label"], axis=1)
            return {k: v.numpy() for k, v in res.items()}

    def load_audio(path: str):
        import torchaudio

        audio, _ = torchaudio.load(path)
        audio = audio.mean(dim=0)
        audio = audio.numpy()
        # audio, _ = librosa.load(path, sr=SAMPLE_RATE)

        target = SAMPLE_RATE * DURATION

        if len(audio) < target:
            audio = np.tile(audio, int(np.ceil(target / len(audio))))

        return audio[:target]

    perch_model = PerchModel(str(PERCH_MODEL_DIR))

    train_audios = pd.read_csv(BIRDCLEF_DIR / "train.csv")["filename"].tolist()
    train_audios = [
        "train_audio/" + t
        for t in train_audios
        if sf.info(BIRDCLEF_DIR / "train_audio" / t).duration <= 60
    ]
    train_soundscape_audios = [
        "train_soundscapes/" + v
        for v in train_soundscapes_labels_df["filename"].unique().tolist()
    ]
    files_to_embed = train_audios + train_soundscape_audios
    print("Files to embed:", len(files_to_embed))

    work = [
        (filepath, i * 5, i * 5 + 5) for filepath in files_to_embed for i in range(12)
    ]

    result = []

    class AudioDataset(torch.utils.data.Dataset):
        def __init__(self, work):
            super().__init__()
            self.work = work
            self.audio = None

        def __len__(self):
            return len(self.work)

        def __getitem__(self, idx: int):
            filepath, start_time, end_time = self.work[idx]

            if self.audio is None or self.audio[0] != filepath:
                self.audio = load_audio(BIRDCLEF_DIR / filepath)
                assert self.audio.shape == (SAMPLE_RATE * DURATION,)

            sr = SAMPLE_RATE
            return self.audio[start_time * sr : end_time * sr]

    ds = AudioDataset(work)
    dl = torch.utils.data.DataLoader(ds, batch_size=64, num_workers=16)

    for batch in tqdm(dl):
        preds = perch_model.predict(batch)
        emb = preds["embedding"]
        assert emb.shape == (len(batch), PERCH_EMBEDDING_DIM)

        result.append(emb)

    result = np.vstack(result)
    embeddings = dict(zip(work, result))

    with open(EMBEDDINGS_CACHE_PATH, "wb") as f:
        pickle.dump(embeddings, f)


with open(EMBEDDINGS_CACHE_PATH, "rb") as f:
    embeddings = pickle.load(f)

embeddings = Embeddings(embeddings)

In [ ]:
from lightning.pytorch.callbacks import (
    LearningRateMonitor,
)
from lightning.pytorch.loggers import WandbLogger

data_module = BirdClefDataModule(
    BIRDCLEF_DIR, embeddings=embeddings, batch_size=config.batch_size
)

training_module = TrainingModule(lr=config.lr)

lr_monitor = LearningRateMonitor(logging_interval="step")
callbacks = [lr_monitor]

logger = WandbLogger(name=run_name)

trainer = L.Trainer(
    max_epochs=config.max_epochs,
    accelerator="auto",
    callbacks=callbacks,
    logger=logger,
    enable_progress_bar=True,
    gradient_clip_val=1.0,
)

In [ ]:
trainer.fit(training_module, data_module)
wandb.finish()